## 0. 学生信息与运行环境

In [4]:
import os, uuid, hashlib, platform, getpass

# ===== 1. 填写学号 =====
STUDENT_ID = 20244080205  

# ===== 2. 打印 conda 环境 =====
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")

# ===== 3. 机器指纹（绑定当前电脑） =====
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]

# ===== 4. 生成唯一 ID =====
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]     {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")


[Conda 环境] base
[学号]     20244080205
[机器指纹] a41d18205864c983
[RUN_ID]   20244080205-a41d18205864c983


# 1. 热身：定义简单函数

* `say_hi`：函数体只有一条 `print` 语句。
* `goat_pad`：用字符串拼接把 `"goat"` 放在前后，再打印。
* `print_n`：用 `for` 循环重复 `n` 次；`print` 默认每次输出后换行，所以自然"每次单独一行"

In [5]:
def say_hi():
    """不接受参数，打印 "Hello, world!"。"""
    print("Hello, world!")


def goat_pad(s):
    """把字符串 s 前后各加上 "goat" 后打印。"""
    print("goat" + s + "goat")


def print_n(s, n):
    """把字符串 s 打印 n 次，每次单独一行。"""
    for _ in range(n):
        print(s)


### 测试

下面依次调用三个函数，检查输出是否符合题目要求。

In [6]:
print("1) say_hi()")
say_hi()

print("\n2) goat_pad('hello')")
goat_pad("hello")

print("\n3) print_n('I love Python!', 4)")
print_n("I love Python!", 4)


1) say_hi()
Hello, world!

2) goat_pad('hello')
goathellogoat

3) print_n('I love Python!', 4)
I love Python!
I love Python!
I love Python!
I love Python!


---

# 2. 欧几里得算法


In [7]:
def gcd(m, n):
    """用欧几里得算法返回 m 和 n 的最大公约数。"""
    while n != 0:
        m, n = n, m % n   # 同时更新：新的 (除数, 余数)
    return m


### 2.2 计算题目给出的三组数对

In [8]:
pairs = [(2018, 2019), (1200, 300), (5040, 60)]

for m, n in pairs:
    print(f"gcd({m}, {n}) = {gcd(m, n)}")


gcd(2018, 2019) = 1
gcd(1200, 300) = 300
gcd(5040, 60) = 60


**结果分析**

| 数对 | GCD | 说明 |
| :--- | :--- | :--- |
| (2018, 2019) | 1 | 两个相邻整数互质 |
| (1200, 300) | 300 | 300 整除 1200 |
| (5040, 60) | 60 | 60 整除 5040 |

三组结果都与手算一致：$2018 = 2 \times 1009$、$2019 = 3 \times 673$，没有公共因子；$1200 = 4 \times 300$；$5040 = 84 \times 60$。


### 2.3 参数为负数时的行为

先实际运行一遍，观察输出：

In [9]:
negative_cases = [(-4, 6), (-4, -6), (4, -6), (6, -4), (-6, 4),
                  (-2018, 2019), (2018, -2019), (0, -5)]

for m, n in negative_cases:
    print(f"gcd({m:>5}, {n:>5}) = {gcd(m, n)}")


gcd(   -4,     6) = 2
gcd(   -4,    -6) = -2
gcd(    4,    -6) = -2
gcd(    6,    -4) = -2
gcd(   -6,     4) = 2
gcd(-2018,  2019) = 1
gcd( 2018, -2019) = -1
gcd(    0,    -5) = -5


**结论：返回值的绝对值仍然正确，但符号可能是负的。**

从上面的输出可以看到规律：

* **第二个参数 `n` 为正**时，结果总是正的，例如 `gcd(-4, 6) = 2`、`gcd(-6, 4) = 2`。
* **第二个参数 `n` 为负**时，结果带负号，例如 `gcd(4, -6) = -2`、`gcd(-4, -6) = -2`、`gcd(2018, -2019) = -1`。

**原因**：Python 的 `%` 是"向下取整"的取模，余数符号与**除数**相同（`4 % (-6) = -2`，而 `(-4) % 6 = 2`）。

* 若 `n > 0`，则 `m % n` 落在 $[0, n)$ 内，第一次循环后两个数都非负，之后不会再出现负数，所以最终返回正数。
* 若 `n < 0`，则 `m % n` 落在 $(n, 0]$ 内，之后每一步的余数都保持非正，最后一个非零余数（即返回值）就是负数。

**这种行为有意义吗？** 部分有意义"：

* 数学上最大公约数定义为**非负**的，`gcd(4, -6)` 的正确结果应当是 `2` 而不是 `-2`。所以严格说这个实现返回的值不总是"最大公约数"，符号不可靠。
* 但从数值上看，返回值的**绝对值**始终等于正确的 GCD，因为负数并不影响"整除"这一性质，算法对 $|m|, |n|$ 的辗转相除过程完全相同。
* 另外 `gcd(0, -5) = -5`、`gcd(0, 0) = 0` 也符合"绝对值正确"的描述（不过 $\gcd(0,0)$ 在数学上没有定义，只是这里算法恰好返回 0）。



---

# 3. 近似欧拉数 $e$

自然对数的底数 $e$ 通常定义为无限和

$$ e = \sum_{k=0}^{\infty} \frac{1}{k!} = 1 + \frac{1}{1!} + \frac{1}{2!} + \frac{1}{3!} + \frac{1}{4!} + \cdots \tag{1} $$

其中 $k!$ 表示 $k$ 的阶乘：$k! = k \cdot (k-1) \cdot (k-2) \cdots 3 \cdot 2 \cdot 1$，并约定 $0! = 1$。


### 3.1 `euler_limit`：用伯努利极限近似 $e$


In [10]:
def euler_limit(n):
    """用 (1 + 1/n)**n 近似 e。"""
    return (1.0 + 1.0 / n) ** n


**验证**：$n$ 越大，$(1+1/n)^n$ 越接近 $e \approx 2.718281828459045$。

In [11]:
import math

TRUE_E = math.exp(1)

print(f"真实值 math.exp(1) = {TRUE_E:.15f}\n")
print(f"{'n':>8}  {'euler_limit(n)':>18}  {'绝对误差':>12}")
for n in [1, 2, 5, 10, 100, 1_000, 10_000, 100_000, 1_000_000]:
    approx = euler_limit(n)
    print(f"{n:>8}  {approx:>18.12f}  {abs(approx - TRUE_E):>12.3e}")


真实值 math.exp(1) = 2.718281828459045

       n      euler_limit(n)          绝对误差
       1      2.000000000000     7.183e-01
       2      2.250000000000     4.683e-01
       5      2.488320000000     2.300e-01
      10      2.593742460100     1.245e-01
     100      2.704813829422     1.347e-02
    1000      2.716923932236     1.358e-03
   10000      2.718145926825     1.359e-04
  100000      2.718268237192     1.359e-05
 1000000      2.718280469096     1.359e-06


### 3.2 `euler_infinite_sum`：用公式 (1) 的部分和近似 $e$


In [12]:
def euler_infinite_sum(n):
    """返回公式(1)前 n 项之和 1/0! + 1/1! + ... + 1/(n-1)!。"""
    total = 0.0
    factorial = 1          # 0! = 1
    for k in range(n):     # k = 0, 1, ..., n-1
        total += 1.0 / factorial
        factorial *= k + 1  # 迭代得到 (k+1)!
    return total


**验证题目中给出的例子**：`euler_infinite_sum(0) == 0`、`(1) == 1`、`(4) ≈ 2.667`。

In [13]:
print(f"euler_infinite_sum(0) = {euler_infinite_sum(0)}")
print(f"euler_infinite_sum(1) = {euler_infinite_sum(1)}")
print(f"euler_infinite_sum(2) = {euler_infinite_sum(2)}")
print(f"euler_infinite_sum(4) = {euler_infinite_sum(4)}  (≈ 2.667)")
print(f"euler_infinite_sum(10) = {euler_infinite_sum(10)}")
print(f"euler_infinite_sum(20) = {euler_infinite_sum(20)}")
print(f"真实值             e = {TRUE_E}")


euler_infinite_sum(0) = 0.0
euler_infinite_sum(1) = 1.0
euler_infinite_sum(2) = 2.0
euler_infinite_sum(4) = 2.6666666666666665  (≈ 2.667)
euler_infinite_sum(10) = 2.7182815255731922
euler_infinite_sum(20) = 2.7182818284590455
真实值             e = 2.718281828459045


部分和收敛得非常快：只要 10 项就已经精确到小数点后 6 位，20 项就达到了双精度浮点数的极限，与 `math.exp(1)` 只差最后一个二进制位。


### 3.3 `euler_approx`：按精度 $\epsilon$ 自动累加


In [14]:
def euler_approx(epsilon):
    """累加公式(1)，直到与 e 的误差小于 epsilon，返回该近似值。"""
    total = 0.0
    term = 1.0     # 第 0 项：1/0! = 1
    k = 0
    while abs(total - TRUE_E) >= epsilon:
        if term == 0.0:
            break       # 已到达浮点精度极限，无法再改进
        total += term
        k += 1
        term /= k       # 递推：1/(k+1)! = (1/k!) / (k+1)
    return total


In [15]:
print(f"{'epsilon':>10}  {'euler_approx':>20}  {'绝对误差':>12}  {'是否达标':>8}")
for epsilon in [1e-1, 1e-3, 1e-6, 1e-10, 1e-15, 1e-16]:
    approx = euler_approx(epsilon)
    error = abs(approx - TRUE_E)
    print(f"{epsilon:>10.0e}  {approx:>20.16f}  {error:>12.3e}  {str(error < epsilon):>8}")

print("\n对照：euler_approx(1e-6) 需要的前 10 项之和 =", euler_infinite_sum(10))


   epsilon          euler_approx          绝对误差      是否达标
     1e-01    2.6666666666666665     5.162e-02      True
     1e-03    2.7180555555555554     2.263e-04      True
     1e-06    2.7182815255731922     3.029e-07      True
     1e-10    2.7182818284467594     1.229e-11      True
     1e-15    2.7182818284590455     4.441e-16      True
     1e-16    2.7182818284590455     4.441e-16     False

对照：euler_approx(1e-6) 需要的前 10 项之和 = 2.7182815255731922


### 3.4 `print_euler_sum_table`：打印累加表


In [16]:
def print_euler_sum_table(n):
    """打印 euler_infinite_sum(1) ... euler_infinite_sum(n)，每行一个。"""
    for k in range(1, n + 1):
        print(euler_infinite_sum(k))


In [17]:
print_euler_sum_table(10)


1.0
2.0
2.5
2.6666666666666665
2.708333333333333
2.7166666666666663
2.7180555555555554
2.7182539682539684
2.71827876984127
2.7182815255731922


从表中能直观地看到部分和的收敛过程：第 1 行是 `1.0`，第 2 行是 `2.0`，第 4 行约为 `2.6667`，第 8 行已经是 `2.71828`，此后与 $e$ 的差距只在浮点末位。


---

# 4. 测试整数属性


In [18]:
def is_power_of_2(n):
    """判断正整数 n 是否为 2 的幂（只用整除和取模）。"""
    while n % 2 == 0:   # 不断除掉因子 2
        n //= 2
    return n == 1       # 只剩 1 说明原来只有因子 2


### 4.1 测试 `is_power_of_2`

In [19]:
test_values = [1, 2, 3, 4, 5, 6, 8, 16, 17, 1024, 1023, 2**10, 2**20]

print(f"{'n':>10}  {'is_power_of_2(n)':>18}")
for n in test_values:
    print(f"{n:>10}  {str(is_power_of_2(n)):>18}")


         n    is_power_of_2(n)
         1                True
         2                True
         3               False
         4                True
         5               False
         6               False
         8                True
        16                True
        17               False
      1024                True
      1023               False
      1024                True
   1048576                True


### 4.2 推广：`is_power(b, n)`


In [20]:
def is_power(b, n):
    """判断正整数 n 是否为 b 的幂。"""
    if b < 1 or n < 1:      # 题目保证输入为正整数，这里仅作保护
        return False
    if b == 1:              # 1 的任何次幂都是 1，避免死循环
        return n == 1
    while n % b == 0:       # 不断除掉因子 b
        n //= b
    return n == 1


In [21]:
cases = [
    (2, 8), (2, 1024), (2, 1023),      # 底数为 2
    (3, 9), (3, 27), (3, 10),          # 底数为 3
    (5, 125), (10, 1000), (10, 999),   # 其他底数
    (4, 16), (4, 8),                   # 4 的幂等价于 2 的偶次幂
    (2, 1), (7, 1),                    # 1 = 任何数的 0 次幂
    (1, 1), (1, 5),                    # b = 1 的边界情况
]

print(f"{'b':>4}  {'n':>6}  {'is_power(b, n)':>16}")
for b, n in cases:
    print(f"{b:>4}  {n:>6}  {str(is_power(b, n)):>16}")


   b       n    is_power(b, n)
   2       8              True
   2    1024              True
   2    1023             False
   3       9              True
   3      27              True
   3      10             False
   5     125              True
  10    1000              True
  10     999             False
   4      16              True
   4       8             False
   2       1              True
   7       1              True
   1       1              True
   1       5             False
